# 標的遺伝子の流れの図（組織の発現つき）: 遺伝子名と組織名を入れると、その組織で発現する遺伝子だけで、上流と下流の遺伝子の選択と図ができる

**使い方**: `config.yaml` の一番上の `gene` に標的遺伝子名を入れて（パラメータとデータベースの場所もすべて `config.yaml`）、上から順に実行する。どんな遺伝子でも、同じ手順で、同じ形の図と表ができる。

## 考え方

### 末端は決めつけない
- これまでの図は、上流の終点を受容体・リガンド、下流の終点を転写因子（TF）と決めていた。この図では**終点を決めつけない**。
- **末端 = これ以上先へ進めない遺伝子**（選んだ基準の辺がもう無い遺伝子）、または最大ステップに着いた遺伝子。受容体・リガンド・TF のこともあれば、キナーゼやそのほかの遺伝子のこともある。
- 受容体・リガンド・TF は、**選ぶ基準には使わない**（上流の受容体だけは、終点として別に足す）。点の色で区別し、**図では端に置く**（下の「表示方法」）。

### 使う数字（どの数字を何に使うか）
| 数字 | 中身 | 使い道 |
|---|---|---|
| **辺の証拠 `E`** | OmniPath の論文数（対数）と SIGNOR のスコアを、ノイジーOR で合成した 0〜1 | 経路スコアの元。辺の太さ |
| **`geo`** | √(`weight` × `E`)（`weight` = データベース内の順位ベースのスコア） | 辺を通すかどうかの基準（`GEO_MIN` 以上） |
| **経路スコア `ps`** | 入力遺伝子からその遺伝子までの**最良経路の辺の `E` の積** | 末端側の遺伝子を選ぶ（`N_END`）。経路の強さ |
| **流れ `flow`** | その遺伝子の先（上流なら外側、下流なら外側）にある**末端の遺伝子すべての `ps` の合計** | **途中の遺伝子を選ぶ唯一の基準**（`N_MID`） |
| 末端の数 `n_end` | `flow` に寄与する末端の遺伝子の数 | 流れの内訳（表に出す） |
| DepMap の相関 `r` | 入力遺伝子との遺伝子効果の相関 | **点の大きさだけ**（選ぶ基準には使わない） |
| 機能 | 役割（リガンド・受容体・TF）と UniProt のキーワード（キナーゼ・ホスファターゼ・ユビキチン経路） | **点の色だけ** |

### 選び方（手順）
1. 入力遺伝子から、`geo ≥ GEO_MIN` の辺を、**上流・下流を同時に**1ホップずつ（最大 `MAX_HOPS`）たどる。**上流は受容体で止める**（受容体より上流は解析しない。入力遺伝子が受容体なら上流へは進まない）。リガンドは解析に使わない。各遺伝子の「親」は、その側で前のホップに着いた遺伝子からの辺のうち `E` が最大のもの（最良経路の木）。**同じホップで両側から着いた遺伝子は、親への辺の `E × weight` が大きい側に 1 回だけ置く**（同点は上流）。
2. 木の各遺伝子の `ps` と、`flow`（先の末端の `ps` の合計）を計算する。
3. **`flow` の大きい順に `N_MID` 個**を選ぶ（上流・下流それぞれ）。遺伝子の `flow` は、その先の遺伝子の `flow` 以上なので、選んだ遺伝子は**必ず入力遺伝子までつながる**。
4. さらに、選んだ遺伝子の隣にある（親が選ばれている）遺伝子のうち、**`ps` の大きい順に `N_END` 個**を足す。これで、証拠の強い末端も見える。上流は、終点の受容体も**`ps` の大きい順に `n_receptors` 個**足す（受容体は先がなく `flow` が小さいため。受容体までの経路の遺伝子も入れる）。最後に、選んだ受容体に作用するリガンド（低分子のリガンドも含む）を `E` の大きい順に受容体ごと最大 `max_ligands_per_receptor` 個、**表示のためだけ**に足す（選ぶ基準には入れない）。

### 表示方法
- 中央に入力遺伝子、左に上流、右に下流。列は入力遺伝子からのステップ数（hop）。遺伝子名は点の上部。
- **端の列**: 上流のリガンドは**左端**、受容体はその右の列、下流の TF は**右端**に置く（先に遺伝子が続くかに関わらず、種類で決める。端に置いた遺伝子の先は描かない。例外は受容体の先のリガンド）。
- **点**: 大きさ = DepMap の相関の大きさ、縁は全部同じ、色 = 機能。**末端の遺伝子（先に描く遺伝子がないもの）の横（上流は左、下流は右）に、流れのスコア `flow`（末端は自分の経路スコア、端の列の遺伝子はその先の末端の合計）の数字を出す**。
- **辺**: 曲線。**薄い青の実線 = 促進、薄い赤の実線（先端は T 字）= 抑制、灰色の点線 = 符号不明**。太さ = 証拠 `E`（0.8 以上 2.0、0.6〜0.8 は 1.6、0.4〜0.6 は 1.2、0.2〜0.4 は 0.9、0.2 未満は 0.6。最大 2）。
- 1 つの列の遺伝子が `wrap_at` を超えるときは、遺伝子数を `wrap_at` で割って切り上げた本数の列に分ける。各列の数は同じくらい（差は 1 以内）。
- **フォーカス遺伝子**（`config.yaml` の `focus.path` のリスト）は、丸を青（太い紺の縁）、名前を紺の太字にする（選ぶ基準には使わない。選ばれなかった遺伝子は図に出ない）。低分子は小さい灰色の四角。凡例は最小限にし、丸の大きさと辺の太さは、図の左下に概要だけを書く。
- 出力は `results/<日時>_<遺伝子名>/` のフォルダにまとめる（図・表・使った設定）。

### 限界
- 流れは「先の末端の `ps` の合計」なので、末端が多いハブ（ユビキチン経路など）が大きく出ることがある。表の `n_end` と、末端の平均の `ps` で内訳を確かめる。
- ネットワークは、ライセンス判定を通った SIGNOR と OmniPath の符号つきの辺だけ（このノートブックの 2〜4）。辺の記録が少ない関係（脂質を介する PI3K → AKT1 など）は、`E` が低く出る。
## このノートブックの位置づけ
`notebooks/network_development.ipynb`（Step 0〜26 の試行錯誤）で**最終的に採用したものだけ**をまとめた、再利用用のノートブック。

| 段階 | 内容 | 採用したもの |
|---|---|---|
| 2〜3 | データベースの読み込みとライセンス判定 | SIGNOR（CC BY 4.0）+ OmniPath。要ライセンス・曖昧なリソースを 1 つでも含む OmniPath の記録は捨てる |
| 4 | ネットワークの統合 | 符号つき・向きつき。重み `weight` = データベース内の順位 + ノイジーOR |
| 5 | 辺の証拠 `E`・`geo`、遺伝子の役割と機能 | `E` = 論文数と SIGNOR スコアのノイジーOR |
| 6〜 | 流れの図 | 上の「選び方」。選択に使わなかった試行（DepMap の相関によるしきい値、ハブごとのまとめ、受容体・TF に限る見方）は含めない |

**この版は、組織の発現（HPA）を使う**: 選んだ組織で発現量が `min_ntpm` 未満のタンパク質を除き、辺の `E`・`weight` に発現量の係数をかける（6b）。SIGNOR の低分子（PIP3・カルシウム・cAMP・ATP など）もノードとして含める（四角の点。選ぶ基準は遺伝子と同じ）。STRING・Reactome FI・DoRothEA・HPA は使わない（STRING は除外、ほかは最終の選択に使わなかった）。

## 1. パラメータ（標的遺伝子を含め、すべて `config.yaml` から読む）
- 動作: `config.yaml` を読み、標的遺伝子名（一番上の `gene`）とパラメータとデータベースの場所を変数にする。値を変えるときは `config.yaml` を編集する
- 出力: 設定の一覧、各データベースのファイルがあるかどうか、**遺伝子記号の確認**（UniProt の遺伝子記号の一覧にない記号は、似た記号を示して止まる。この時点では出力フォルダを作らない）。出力は日付のフォルダの中の、実行ごとのフォルダ `results/<日付>/<時刻>_<遺伝子名>/` にまとめる（図・表・使った設定とフォーカスリストの写し）。`config.yaml` の `focus.path` にフォーカス遺伝子のリストを書くと、図でその遺伝子を目立たせる（`null` なら無し）

In [ ]:
# config.yaml を読む（パラメータとデータベースの場所はすべてここ）
import os, re, json, math, yaml
import numpy as np, pandas as pd, plotly.graph_objects as go
from collections import defaultdict, Counter

CONFIG = "config.yaml"                                   # このノートブックと同じフォルダ
cfg = yaml.safe_load(open(CONFIG))
GENE = cfg["gene"].upper()                                      # 標的遺伝子名は config.yaml の一番上
symbols = set(pd.read_csv(os.path.expanduser(cfg["databases"]["uniprot"]["names"]), sep="	", usecols=["Gene Names (primary)"]).dropna()["Gene Names (primary)"].str.upper())   # 遺伝子記号の一覧（UniProt reviewed ヒト）
if GENE not in symbols:                                                                      # 解析やフォルダ作りの前に、遺伝子記号が正しいか確かめる
    import difflib
    raise SystemExit(f"[遺伝子記号の確認] {GENE} は遺伝子記号として見つからない。似た記号: {', '.join(difflib.get_close_matches(GENE, sorted(symbols), n=5, cutoff=0.6)) or 'なし'}（別名や綴りの誤りでないか確かめて、config.yaml の gene を直す）")
print(f"[遺伝子記号の確認] {GENE}: OK")
tis_all = sorted(set(pd.read_csv(os.path.expanduser(cfg["databases"]["hpa_tissue"]["path"]), sep="	", usecols=["Tissue"]).Tissue))   # HPA の組織名の一覧
TISSUE = {t.lower(): t for t in tis_all}.get(str(cfg["tissue"]["name"]).lower().strip())    # 組織名（大文字小文字は問わない）
if TISSUE is None:
    import difflib
    raise SystemExit(f"[組織名の確認] {cfg['tissue']['name']} は HPA の組織名にない。似た組織: {', '.join(difflib.get_close_matches(str(cfg['tissue']['name']).lower(), tis_all, n=5, cutoff=0.4)) or 'なし'}。全部: {', '.join(tis_all)}")
MIN_NTPM, REF_NTPM, UNKNOWN = cfg["tissue"]["min_ntpm"], cfg["tissue"]["ref_ntpm"], cfg["tissue"]["unknown"]
print(f"[組織名の確認] {TISSUE}: OK | min_ntpm={MIN_NTPM} | ref_ntpm={REF_NTPM} | unknown={UNKNOWN}")
GEO_MIN, MAX_HOPS, N_MID, N_END, MAX_OUT = (cfg["flow"][k] for k in ("geo_min", "max_hops", "n_mid", "n_end", "max_out"))
SIGNOR_TYPES = [t.lower() for t in cfg["databases"]["signor"]["entity_types"]]
MAX_CX = cfg["databases"]["signor"]["max_complex_members"]                                  # これより大きい複合体は展開しない   # SIGNOR で辺の端点にしてよい種類（タンパク質 + 低分子）
N_REC = cfg["flow"]["n_receptors"]
FOCUS_PATH = (cfg.get("focus") or {}).get("path")                                          # フォーカス遺伝子のリスト（1 行に 1 遺伝子。# で始まる行は無視。CSV なら 1 列目）
FOCUS = set()
if FOCUS_PATH: FOCUS = {l.split(",")[0].split("\t")[0].strip().upper() for l in open(os.path.expanduser(FOCUS_PATH), encoding="utf-8-sig") if l.strip() and not l.lstrip().startswith("#")}
WRAP_AT, ROW_PX = cfg["figure"]["wrap_at"], cfg["figure"]["row_px"]
from datetime import datetime
import shutil
NOW = datetime.now()
RUN = f"{NOW:%H%M%S}_{GENE}_{TISSUE.replace(' ', '_')}"                                                       # 実行の時刻と入力遺伝子
OUT = os.path.join(cfg["output"]["dir"], f"{NOW:%Y%m%d}", RUN)                                  # フォルダは、遺伝子記号とネットワークの確認のあとに作る（ステップ 5）。 出力は日付のフォルダの中の、実行ごとのフォルダにまとめる: results/<日付>/<時刻>_<遺伝子>/

DB = {"signor": cfg["databases"]["signor"]["path"], "omnipath": cfg["databases"]["omnipath"]["path"], "omnipath_resources": cfg["databases"]["omnipath"]["resources"],
      "uniprot": cfg["databases"]["uniprot"]["path"], "tf_list": cfg["databases"]["tf_list"]["path"], "depmap": cfg["databases"]["depmap"]["dir"]}
DB = {k: os.path.expanduser(v) for k, v in DB.items()}
print(f"GENE={GENE} | geo>={GEO_MIN} | max_hops={MAX_HOPS} | n_mid={N_MID} | n_end={N_END} | max_out={MAX_OUT}")
pd.DataFrame({"path": DB, "exists": {k: os.path.exists(v) for k, v in DB.items()}})

## 2. データベースの読み込み
- 動作: SIGNOR、OmniPath（ライセンスフィルタ前の生ファイル）とそのリソース別ライセンス一覧、UniProt、転写因子一覧を読む
- 出力: 各表の行数

In [ ]:
# データベースを読む
signor = pd.read_csv(DB["signor"], sep="\t", dtype=str, quoting=3, on_bad_lines="skip",
                     usecols=["ENTITYA", "TYPEA", "ENTITYB", "TYPEB", "EFFECT", "MECHANISM", "RESIDUE", "TAX_ID", "DIRECT", "SCORE"])
omni = pd.read_csv(DB["omnipath"], sep="\t", dtype=str,
                   usecols=["source_genesymbol", "target_genesymbol", "is_directed", "is_stimulation", "is_inhibition", "consensus_stimulation", "consensus_inhibition", "sources", "references"])
res = json.load(open(DB["omnipath_resources"]))                           # OmniPath のリソース別ライセンス
names_tab = pd.read_csv(os.path.expanduser(cfg["databases"]["uniprot"]["names"]), sep="	", usecols=["Gene Names (primary)", "Protein names"]).dropna()   # 遺伝子名（タンパク質名）の表
uniprot = pd.read_csv(DB["uniprot"], sep="\t", usecols=["Gene Names (primary)", "Keywords", "Subcellular location [CC]"]).dropna(subset=["Gene Names (primary)"])
tf_tab = pd.read_csv(DB["tf_list"], usecols=["HGNC symbol", "Is TF?"])
print({"SIGNOR": len(signor), "OmniPath": len(omni), "OmniPath resources": len(res), "UniProt": len(uniprot), "TF list": len(tf_tab)})

## 3. OmniPath のライセンス判定（厳しい基準）
- 動作: OmniPath の記録が載せているリソースを 1 つずつ「要ライセンス / 不要 / 曖昧」に分け、要ライセンスまたは曖昧なリソースを 1 つでも含む記録を捨てる（`config.yaml` の `license_strict`）
- 出力: 区分ごとのリソース数、残った記録の数

In [ ]:
# リソースごとのライセンス区分を決め、全リソースが「不要」の記録だけ残す
CLEAR, SHARE = set(cfg["license_strict"]["clear"]), set(cfg["license_strict"]["share_alike"])
count = Counter(s for x in omni.sources.dropna() for s in x.split(";") if s)          # リソースごとの記録数
cls = {}
for s in count:
    l = (res.get(s) or res.get(s.split("_")[0]) or {}).get("license") or {}           # リソース名がそのまま一覧になければ "_" の前の名前
    p, n = l.get("purpose"), l.get("name")
    cls[s] = "要ライセンス" if p in ("academic", "non_profit", "nonprofit") else "不要" if (p == "commercial" and n in CLEAR) else "不要（共有条件あり）" if (p == "commercial" and n in SHARE) else "曖昧"
src_list = omni.sources.fillna("").map(lambda x: [s for s in x.split(";") if s])
use = src_list.map(lambda l: all(cls[s].startswith("不要") for s in l))               # 全部「不要」のときだけ残す
omni_strict = omni[use].assign(sources_ok=src_list[use])
print(pd.Series(Counter(cls.values()), name="リソース数").to_frame().T.to_string())
print(f"OmniPath の記録: {len(omni)} → 残す {len(omni_strict)}（捨てる {len(omni) - len(omni_strict)}）")

## 4. 主シグナル伝達ネットワークの構築
- 動作: SIGNOR（ヒト・直接の辺。端点はタンパク質と低分子。複合体は構成タンパク質に展開する: メンバーは SIGNOR の「form complex」の行から作る。タンパク質ファミリーは展開しない）と OmniPath（向きと符号のある記録、複合体は構成タンパク質に展開）を、符号つきの 1 つのネットワークにまとめる。重み `weight` は、データベース内の順位（パーセンタイル）を 0〜1 にして、データベース間をノイジーOR で合成する
- 出力: 辺の数・ノード数、辺の表の先頭

In [ ]:
# SIGNOR + OmniPath(厳しい基準) -> 符号つき・向きつきネットワーク
sg_h = signor[(signor.TAX_ID.astype(str).str.strip() == "9606") & (signor.DIRECT.str.upper() == "YES")]
tl = lambda c: sg_h[c].str.lower()
is_form = sg_h.EFFECT.str.lower().str.contains("form complex", na=False)                 # 「複合体を作る」の行（メンバーの情報。辺としては使わない）
# --- 複合体のメンバー（form complex の行から作る。複合体の中の複合体も、タンパク質まで展開する） ---
mem = defaultdict(set)
for a_, ta_, b_ in zip(sg_h.ENTITYA[is_form & (tl("TYPEB") == "complex")].str.upper().str.strip(), tl("TYPEA")[is_form & (tl("TYPEB") == "complex")], sg_h.ENTITYB[is_form & (tl("TYPEB") == "complex")].str.upper().str.strip()):
    if ta_ in ("protein", "complex"): mem[b_].add(a_)
def expand(name, seen=()):                                                              # 複合体 -> 構成タンパク質の集合（再帰）。メンバーが分からない複合体は空
    if name not in mem: return set()
    out = set()
    for m_ in mem[name]:
        out |= expand(m_, seen + (name,)) if (m_ in mem and m_ not in seen) else ({m_} if m_ not in mem else set())
    return out
cx_prot = {c: expand(c) for c in mem}
# --- 辺: タンパク質・低分子どうしはそのまま、複合体の端点は構成タンパク質に展開する ---
ok_t = lambda c: tl(c).isin(SIGNOR_TYPES + ["complex"])
sg = sg_h[ok_t("TYPEA") & ok_t("TYPEB") & ~is_form]
eff = sg.EFFECT.str.lower().map(lambda e: 1 if "up-regulates" in e else -1 if "down-regulates" in e else 0)
rows_all = pd.DataFrame({"src": sg.ENTITYA.str.upper().str.strip(), "dst": sg.ENTITYB.str.upper().str.strip(), "ta": sg.TYPEA.str.lower(), "tb": sg.TYPEB.str.lower(), "effect": eff,
                         "mechanism": sg.MECHANISM.fillna(""), "residue": sg.RESIDUE.fillna(""), "source": "SIGNOR", "score": pd.to_numeric(sg.SCORE, errors="coerce")})
is_cx_row = (rows_all.ta == "complex") | (rows_all.tb == "complex")
rows = []; n_cx_rows = n_no_member = n_too_big = 0
for r in rows_all[is_cx_row].itertuples():
    n_cx_rows += 1
    A = cx_prot.get(r.src, set()) if r.ta == "complex" else {r.src}
    B = cx_prot.get(r.dst, set()) if r.tb == "complex" else {r.dst}
    if not A or not B: n_no_member += 1; continue                                       # メンバーが分からない複合体の辺は捨てる
    if (len(A) > MAX_CX or len(B) > MAX_CX): n_too_big += 1; continue                   # 大きすぎる複合体（リボソームなど）は展開しない
    rows += [dict(src=a, dst=b, effect=r.effect, mechanism=r.mechanism + ";complex", residue=r.residue, source="SIGNOR", score=r.score) for a in A for b in B]
edges_sg = pd.concat([rows_all[~is_cx_row].drop(columns=["ta", "tb"]), pd.DataFrame(rows)], ignore_index=True)
print(f"SIGNOR の複合体: {len(mem)}（メンバーが分かる {sum(bool(v) for v in cx_prot.values())}）| 複合体を含む辺 {n_cx_rows} → 展開した辺 {len(rows)}（メンバー不明で捨てた {n_no_member}、大きすぎて捨てた {n_too_big}）")
edges_sg = edges_sg[(edges_sg.src != edges_sg.dst) & (edges_sg.src != "") & (edges_sg.dst != "")]
edges_sg = edges_sg.sort_values(["effect", "score"], key=lambda s: (s == 0) if s.name == "effect" else -s.fillna(0)).drop_duplicates(["src", "dst"])

om = omni_strict[omni_strict.is_directed.isin(["True", "true", "1"])].copy()
T = lambda col: om[col].isin(["True", "true", "1"])
stim, inh, cstim, cinh = T("is_stimulation"), T("is_inhibition"), T("consensus_stimulation"), T("consensus_inhibition")
om["effect"] = 0
om.loc[(stim & ~inh) | (cstim & ~cinh), "effect"] = 1
om.loc[(inh & ~stim) | (cinh & ~cstim), "effect"] = -1
om = om[stim | inh]                                                                      # 符号の情報がある記録だけ
e = pd.DataFrame({"src": om.source_genesymbol.str.upper(), "dst": om.target_genesymbol.str.upper(), "effect": om.effect,
                  "mechanism": "omnipath:" + om.sources_ok.map(lambda l: "|".join(sorted(set(l)))), "residue": "", "source": "OmniPath",
                  "score": om.references.fillna("").map(lambda r: len([x for x in r.split(";") if x]))})      # 元のスコア = 論文の数
members = lambda x: [m for m in x.replace("COMPLEX:", "").split("_") if m]               # COMPLEX:A_B -> [A, B]
is_cx = e.src.str.contains("_") | e.dst.str.contains("_")
rows = [dict(src=a, dst=b, effect=r.effect, mechanism=r.mechanism + ";complex", residue="", source="OmniPath", score=r.score) for r in e[is_cx].itertuples() for a in members(r.src) for b in members(r.dst)]
e = pd.concat([e[~is_cx], pd.DataFrame(rows)], ignore_index=True)
edges_om = e[(e.src != e.dst) & (e.src != "") & (e.dst != "")].sort_values(["effect", "score"], key=lambda s: (s == 0) if s.name == "effect" else -s).drop_duplicates(["src", "dst"])

def merge_group(g):                                                                      # 同じ (src, dst) の記録を 1 行にまとめる（符号が割れたら 0 = 不明）
    ef = g.effect[g.effect != 0]
    return pd.Series(dict(effect=0 if len(set(ef)) != 1 else int(ef.iloc[0]), mechanism=";".join(sorted({m for m in g.mechanism if m})),
                          residue=";".join(sorted({r for r in g.residue if r})), source="+".join(sorted(set(g.source)))))
a, b = edges_sg.copy(), edges_om.copy()
for d in (a, b): d["nscore"] = d.score.rank(pct=True).fillna(0.5)                       # データベース内の順位 -> 0〜1
allr = pd.concat([a, b], ignore_index=True)
net = allr.groupby(["src", "dst"], sort=False).apply(merge_group, include_groups=False)
per = allr.groupby(["src", "dst", "source"], sort=False).nscore.max().reset_index()
per["l"] = np.log1p(-per.nscore.clip(upper=0.999))
net["weight"] = (1 - np.exp(per.groupby(["src", "dst"], sort=False).l.sum())).round(3)  # ノイジーOR
net = net.reset_index()
print(f"辺 {len(net)} | ノード {len(set(net.src) | set(net.dst))} | 符号: 促進 {int((net.effect == 1).sum())} / 抑制 {int((net.effect == -1).sum())} / 不明 {int((net.effect == 0).sum())}")
net.head()

## 5. 辺のスコア（`E`、`geo`）
- 動作: 辺の証拠 `E` = 論文数（対数、最大で割る）と SIGNOR のスコアのノイジーOR。`geo` = √(`weight` × `E`)。`geo` が `geo_min` 以上の辺だけを、以降の探索に通す
- 出力: 各スコアの分布、辺の数

In [ ]:
# 辺の証拠 E と geo
oe = edges_om.set_index(["src", "dst"]).score; sgs = edges_sg.set_index(["src", "dst"]).score        # OmniPath の論文数 / SIGNOR のスコア
key = pd.MultiIndex.from_frame(net[["src", "dst"]])
edges = net.copy()
edges["refs"] = oe.reindex(key).values; edges["signor"] = sgs.reindex(key).values
A = np.log1p(edges.refs.fillna(0)) / np.log1p(oe.max()); B = edges.signor.fillna(0)
edges["E"] = (1 - (1 - A) * (1 - B)).round(4)
edges["geo"] = np.sqrt(edges.weight * edges.E).round(4)
edges["mechanism"] = edges.mechanism.astype(str).str[:90]
if GENE not in set(edges.src) | set(edges.dst): raise SystemExit(f"[ネットワークの確認] {GENE} は遺伝子記号としては正しいが、ネットワーク（SIGNOR・OmniPath）に辺がない")
os.makedirs(OUT, exist_ok=True)                                                              # 確認が通ってから、出力フォルダを作る
shutil.copy(CONFIG, os.path.join(OUT, "config_used.yaml"))                                  # 使った設定（パラメータとデータベースの場所）を残す
if FOCUS_PATH: shutil.copy(os.path.expanduser(FOCUS_PATH), os.path.join(OUT, "focus_genes_used.txt"))
print(f"[出力] {OUT}")
ok_edges = edges[edges.geo >= GEO_MIN]
print(f"辺 {len(edges)}（geo >= {GEO_MIN}: {len(ok_edges)}）| {GENE} から出る辺 {int((ok_edges.src == GENE).sum())}、入る辺 {int((ok_edges.dst == GENE).sum())}")
edges[["effect", "refs", "signor", "weight", "E", "geo"]].describe().round(3)

## 5b. 代謝の辺を足す（Human-GEM）
- 動作: 代謝モデル Human-GEM の反応から、**酵素の遺伝子と代謝物**の辺を足す。基質 → 酵素、酵素 → 産物（反応の書かれた向きだけ。可逆な反応の逆向きは `use_reversible: true` のときだけ使う）。符号は不明（0）。水・H+・ATP・NADPH など多くの反応に出る代謝物（`max_reactions_per_metabolite` 反応より多いもの）は、すべてをつないでしまうので除く。区画の違い（細胞質・ミトコンドリアなど）は同じ代謝物にまとめ、基質と産物が同じ反応（膜輸送）は使わない。辺のスコアは `config.yaml` の `metabolism` の固定値（`edge_E`、`edge_weight`）。SIGNOR にすでにある辺は上書きしない
- 出力: 足した辺と代謝物の数、除いた代謝物の数。代謝物は図で小さい灰色の四角になる

In [ ]:
# Human-GEM の代謝の辺（酵素 <-> 基質・産物）を足す
from collections import Counter
MET = cfg.get("metabolism") or {}
met_names = set()
if MET.get("enabled"):
    gem = dict(yaml.load(open(os.path.expanduser(cfg["databases"]["human_gem"]["path"])), Loader=yaml.CSafeLoader))   # Human-GEM.yml（反応・代謝物・遺伝子の対応）
    mname = {dict(m)["id"]: str(dict(m)["name"]).upper().strip() for m in gem["metabolites"]}                         # 代謝物の ID -> 名前（区画は名前にまとめる）
    gtab = pd.read_csv(os.path.expanduser(cfg["databases"]["human_gem"]["genes"]), sep="\t", usecols=["genes", "geneSymbols"]).dropna()
    sym = dict(zip(gtab.genes, gtab.geneSymbols.str.upper()))                                                         # Ensembl ID -> 遺伝子記号
    rx = [dict(r) for r in gem["reactions"]]
    nrx = Counter(n for r in rx for n in {mname[m] for m, _ in r["metabolites"]})
    hub = {n for n, c in nrx.items() if c > MET["max_reactions_per_metabolite"]}                                      # 水・H+・ATP・NADPH など（すべてをつなぐので除く）
    rows = []
    for r in rx:
        enz = {sym[g] for g in re.findall(r"ENSG\d+", str(r.get("gene_reaction_rule", ""))) if g in sym}
        if not enz: continue
        sub = {mname[m] for m, c in r["metabolites"] if c < 0} - hub
        prod = {mname[m] for m, c in r["metabolites"] if c > 0} - hub
        s_only, p_only = sub - prod, prod - sub                                                                       # 基質と産物が同じ（膜輸送）は使わない
        mech = f"metabolic reaction {r['id']} ({(r.get('subsystem') or [''])[0]})"[:90]
        rev = MET.get("use_reversible", False) and r.get("lower_bound", 0) < 0                                         # 可逆な反応の逆向きは、設定したときだけ使う（使うと上流・下流が同じ代謝物につながって区別できなくなる）
        for e_ in enz:
            rows += [(s, e_, mech) for s in s_only] + [(e_, p, mech) for p in p_only]
            if rev: rows += [(e_, s, mech) for s in s_only] + [(p, e_, mech) for p in p_only]
    met = pd.DataFrame(rows, columns=["src", "dst", "mechanism"]).drop_duplicates(["src", "dst"])
    met = met[~(met.src + "|" + met.dst).isin(edges.src + "|" + edges.dst)]                                           # SIGNOR・OmniPath の辺は上書きしない
    met = met.assign(effect=0, residue="", source="Human-GEM", weight=MET["edge_weight"], refs=np.nan, signor=np.nan, E=MET["edge_E"])
    met["geo"] = np.sqrt(met.weight * met.E).round(4)
    met_names = (set(met.src) | set(met.dst)) - set(sym.values())                                                      # 代謝物の名前（遺伝子は除く）
    edges = pd.concat([edges, met[edges.columns]], ignore_index=True)
    ok_edges = edges[edges.geo >= GEO_MIN]
    print(f"代謝の辺 {len(met)}（酵素 {len(set(met.src) | set(met.dst)) - len(met_names)} 遺伝子、代謝物 {len(met_names)}）| 除いた代謝物（{MET['max_reactions_per_metabolite']} 反応より多い）{len(hub)}: " + ", ".join(sorted(hub, key=lambda n: -nrx[n])[:8]) + " ...")
    print(f"全体: 辺 {len(edges)}（geo >= {GEO_MIN}: {len(ok_edges)}）| {GENE} から出る辺 {int((ok_edges.src == GENE).sum())}、入る辺 {int((ok_edges.dst == GENE).sum())}")

## 6. 遺伝子の役割と機能（色分け用）
- 動作: SIGNOR の低分子（smallmolecule = 代謝物・脂質・イオンなど、chemical = 化合物・薬剤）はそのまま `small molecule` / `chemical` にする（タンパク質と同名のものはタンパク質）。UniProt のキーワード・局在から受容体とリガンド（分泌された Cytokine / Growth factor / Hormone）を決め、転写因子一覧と合わせて役割を付ける。そのほかはキナーゼ・ホスファターゼ・ユビキチン経路・その他に分ける。選ぶ基準には使わず、点の色と端の列の判定だけに使う
- 出力: 機能ごとの遺伝子の数

In [ ]:
# 遺伝子の機能（ligand > receptor > transcription factor > kinase > phosphatase > ubiquitin pathway > other）と局在
u = uniprot.assign(g=uniprot["Gene Names (primary)"].str.upper()).drop_duplicates("g").set_index("g")
kw, loc = u.Keywords.fillna(""), u["Subcellular location [CC]"].fillna("")
receptors = set(u.index[kw.str.contains("Receptor") & loc.str.contains("Cell membrane")])
ligands = set(u.index[loc.str.contains("Secreted") & kw.str.contains("Cytokine|Growth factor|Hormone")]) - receptors
tfs = set(tf_tab.loc[tf_tab["Is TF?"] == "Yes", "HGNC symbol"].str.upper()) - ligands
receptors -= tfs                                                                          # 重なりは TF を優先
uni_cat = pd.Series("other", index=u.index)                                              # UniProt のキーワードによる機能（弱い順に上書き）
uni_cat[kw.str.contains("Ubl conjugation pathway")] = "ubiquitin pathway"
uni_cat[kw.str.contains("Protein phosphatase")] = "phosphatase"
uni_cat[kw.str.contains("Kinase")] = "kinase"
places = {"membrane": "membrane", "nucleus": "nucleus", "cytoplasm|cytosol": "cytoplasm", "secreted": "secreted", "mitochondri": "mitochondrion", "endoplasmic reticulum|golgi": "ER/Golgi", "lysosome": "lysosome"}
flags = {"Kinase": "kinase", "Protein phosphatase": "phosphatase", "Ubl conjugation pathway": "ubiquitin_pathway", "Apoptosis": "apoptosis", "Cell cycle": "cell_cycle", "Transcription": "transcription", "Receptor": "receptor"}
loc_txt = loc.map(lambda t: ";".join(v for k, v in places.items() if re.search(k, t, re.I)))
flag_txt = kw.map(lambda t: ";".join(v for k, v in flags.items() if k in t))
sg_all = signor[signor.TAX_ID.astype(str).str.strip() == "9606"]                          # SIGNOR の低分子: 名前 -> 種類（smallmolecule = 代謝物・脂質・イオンなど、chemical = 化合物・薬剤）
sm_type = {}
for t_col, n_col in (("TYPEA", "ENTITYA"), ("TYPEB", "ENTITYB")):
    for t, n in zip(sg_all[t_col].str.lower(), sg_all[n_col].str.upper().str.strip()):
        if t in ("smallmolecule", "chemical"): sm_type[n] = t
prot_names = set(pd.concat([sg_all.ENTITYA[sg_all.TYPEA.str.lower() == "protein"], sg_all.ENTITYB[sg_all.TYPEB.str.lower() == "protein"]]).str.upper().str.strip())
both_names = set(sm_type) & prot_names                                                    # タンパク質と同じ名前の低分子（タンパク質として扱う）
sm_type = {n: t for n, t in sm_type.items() if n not in prot_names}
print(f"低分子 {len(sm_type)}（smallmolecule {sum(t == 'smallmolecule' for t in sm_type.values())} / chemical {sum(t == 'chemical' for t in sm_type.values())}）| タンパク質と同名で除外 {len(both_names)}")
gname = dict(zip(names_tab["Gene Names (primary)"].str.upper(), names_tab["Protein names"].str.split(" \(", n=1).str[0]))   # 遺伝子記号 -> 遺伝子名（最初の名前。別名や EC 番号の括弧は除く）
glist = sorted(set(edges.src) | set(edges.dst))
genes = pd.DataFrame({"gene": glist,
                      "func": ["small molecule" if (sm_type.get(g) == "smallmolecule" or g in met_names) else "chemical" if sm_type.get(g) == "chemical" else "ligand" if g in ligands else "receptor" if g in receptors else "transcription factor" if g in tfs else uni_cat.get(g, "other") for g in glist],
                      "gene_name": [gname.get(g, "") for g in glist], "locations": [loc_txt.get(g, "") for g in glist], "uniprot_flags": [flag_txt.get(g, "") for g in glist]}).set_index("gene")
print(genes.func.value_counts().to_dict())

## 6b. 組織の発現で、遺伝子を絞り、辺の重みに発現量をかける（Human Protein Atlas）
- 動作: HPA の組織別の発現量（nTPM）を読み、選んだ組織（`config.yaml` の `tissue.name`）で、次の 2 つを行う。
  1. **遺伝子を絞る**: 発現量が `min_ntpm` 未満のタンパク質を、ネットワークから除く（辺も消える）。代謝物・低分子は発現のデータがないので除かない。HPA にないタンパク質は `unknown`（`keep` で残す、`drop` で除く）。入力遺伝子は、発現が低くても残し、警告を出す。
  2. **辺の重みに発現量をかける**: 遺伝子ごとの係数 `f = min(1, log(1+nTPM) / log(1+ref_ntpm))`（`ref_ntpm` 以上の発現で 1）。辺の係数は両端の `f` の幾何平均 √(f_src × f_dst)。辺の `E` と `weight` の両方にかけるので、`geo` は係数倍になり、経路スコア・流れにも反映される。
- あわせて、組織特異性 `tau`（0 = どの組織でも同じ、1 = 1 つの組織だけ）を表に付ける（選ぶ基準には使わない）。
- 出力: 絞る前後の遺伝子・辺の数、入力遺伝子の発現量。ライセンス: HPA は CC BY-SA 3.0（フィルタと重みにだけ使う。表に出る nTPM は、再配布するなら同じ条件で共有する）

In [ ]:
# 組織の発現（HPA）で遺伝子を絞り、辺の重みに発現量をかける
hpa = pd.read_csv(os.path.expanduser(cfg["databases"]["hpa_tissue"]["path"]), sep="\t", usecols=["Gene name", "Tissue", "nTPM"])
wide = hpa.assign(g=hpa["Gene name"].str.upper()).pivot_table(index="g", columns="Tissue", values="nTPM", aggfunc="max")   # 遺伝子 × 組織の発現量
ntpm = wide[TISSUE]                                                                          # 選んだ組織の発現量
tau = ((1 - wide.div(wide.max(axis=1).replace(0, np.nan), axis=0)).sum(axis=1) / (wide.shape[1] - 1)).round(3)   # 組織特異性
genes["ntpm"] = genes.index.map(ntpm); genes["tau"] = genes.index.map(tau)
in_hpa = genes.index.isin(wide.index)                                                        # 発現のデータがある遺伝子
is_met = genes.func.isin(["small molecule", "chemical"])                                      # 代謝物・低分子は発現のデータがない
keep_g = pd.Series(np.where(in_hpa, genes.ntpm.fillna(0) >= MIN_NTPM, is_met | (UNKNOWN == "keep")), index=genes.index)
keep_g[GENE] = True                                                                          # 入力遺伝子は残す
fac = pd.Series(np.where(in_hpa, np.minimum(1, np.log1p(genes.ntpm.fillna(0)) / np.log1p(REF_NTPM)), 1.0), index=genes.index)   # 発現量の係数
n_e0, n_g0 = len(edges), len(genes)
edges = edges[edges.src.map(keep_g).fillna(False) & edges.dst.map(keep_g).fillna(False)].copy()   # 両端が残る辺だけ
edge_f = np.sqrt(edges.src.map(fac) * edges.dst.map(fac))                                    # 辺の係数 = 両端の係数の幾何平均
edges["E"] = (edges.E * edge_f).round(4); edges["weight"] = (edges.weight * edge_f).round(4)
edges["geo"] = np.sqrt(edges.weight * edges.E).round(4)
ok_edges = edges[edges.geo >= GEO_MIN]
used = set(edges.src) | set(edges.dst)
print(f"組織: {TISSUE} | 遺伝子（ノード）{n_g0} → {len(used)} | 辺 {n_e0} → {len(edges)}（geo >= {GEO_MIN}: {len(ok_edges)}）")
print(f"{GENE}: nTPM {genes.ntpm.get(GENE, float('nan')):.1f}（係数 {fac[GENE]:.2f}、組織特異性 tau {genes.tau.get(GENE, float('nan'))}）" + ("  ← 警告: 発現が基準（min_ntpm）未満" if (genes.ntpm.get(GENE, 0) or 0) < MIN_NTPM else ""))
print(f"{GENE} から出る辺 {int((ok_edges.src == GENE).sum())}、入る辺 {int((ok_edges.dst == GENE).sum())}（組織で絞ったあと）")

## 7. DepMap の相関（点の大きさ用）
- 動作: 入力遺伝子と、ネットワークの各遺伝子との CRISPR 遺伝子効果の相関（ピアソン）を計算する。**点の大きさだけ**に使い、遺伝子の選択には使わない
- 出力: 相関を計算した遺伝子の数

In [ ]:
# 入力遺伝子との DepMap の相関（点の大きさに使う）
eff_f = os.path.join(DB["depmap"], "CRISPRGeneEffect.csv")
hdr = pd.read_csv(eff_f, nrows=0).columns
colmap = {c.split(" (")[0].upper(): c for c in hdr if " (" in c}
if GENE in colmap:
    use_g = (set(edges.src) | set(edges.dst)) & set(colmap) | {GENE}
    dm = pd.read_csv(eff_f, usecols=[colmap[g] for g in use_g], dtype=np.float32)
    dm.columns = [c.split(" (")[0].upper() for c in dm.columns]                           # usecols は列をファイルの順に返すので、名前は読んだ列から付ける
    corr = dm.corrwith(dm[GENE]).drop(GENE)
    print(f"相関を計算した遺伝子: {len(corr)}（細胞株 {len(dm)}）")
else:
    corr = pd.Series(dtype=float); print(f"{GENE} は DepMap にない → 点の大きさは全部最小")

## 8. 上流・下流へ1ホップずつたどる
- 動作: 入力遺伝子から `geo >= geo_min` の辺を、上流・下流**同時に**1ホップずつ（最大 `max_hops`）たどる（遺伝子の種類では止めない。ただし**上流は受容体で止める**: 受容体より上流は解析しない。リガンドも解析には使わない）。各遺伝子の親は、その側で前のホップに着いた遺伝子からの辺のうち `E` が最大のもの。**同じホップで両側から着いた遺伝子は、親への辺の `E × weight` が大きい側に置く**（同点は上流）。先に着いた側（ホップが小さい側）が優先で、一度置いた遺伝子は反対側に出さない。上流・下流とも、置いた側の遺伝子を通る経路だけで木ができるので、「途中の遺伝子が反対側に取られて経路が切れる」ことがない
- 出力: 同じホップで両側から着いた遺伝子の数と例、hop ごとの遺伝子の数

In [ ]:
# 上流・下流の探索（遺伝子の種類で止めない）。上流・下流を同時に 1 ホップずつ進め、同じホップで両側から着いた遺伝子は E × weight が大きい側に置く
inc = {"down": dict(list(ok_edges.groupby("src"))), "up": dict(list(ok_edges.groupby("dst")))}
far_col = {"up": "src", "down": "dst"}                                                      # 入力遺伝子から見て遠い側の遺伝子の列
claimed = {GENE: "seed"}                                                                    # 遺伝子 -> 置いた側
front = {"up": [GENE], "down": [GENE]}
rec = {"up": [], "down": []}
contested = []                                                                              # 同じホップで両側から着いた遺伝子の記録
for h in range(1, MAX_HOPS + 1):
    cand = {"up": {}, "down": {}}
    for side in ("up", "down"):
        for v in front[side]:
            if v not in inc[side]: continue
            if side == "up" and v in receptors: continue                                    # 受容体より上流は解析しない（入力遺伝子が受容体のときも、上流へは進まない）
            g_ = inc[side][v]
            g_ = g_[~g_[far_col[side]].isin(ligands)] if side == "up" else g_              # リガンドは解析に使わない（受容体の隣に、表示のためだけに足す: 次のセル）
            for r in g_[~g_[far_col[side]].isin(claimed)].itertuples():
                cand[side].setdefault(getattr(r, far_col[side]), []).append(dict(hop=h, gene=getattr(r, far_col[side]), parent=v, effect=r.effect, E=r.E, weight=r.weight, geo=r.geo, mechanism=r.mechanism))
    front = {"up": [], "down": []}
    for g in set(cand["up"]) | set(cand["down"]):
        best = {s_: max(cand[s_][g], key=lambda d: d["E"]) for s_ in ("up", "down") if g in cand[s_]}   # 各側で E が最大の辺を親にする
        if len(best) == 2:
            s_up, s_down = best["up"]["E"] * best["up"]["weight"], best["down"]["E"] * best["down"]["weight"]
            side = "up" if s_up >= s_down else "down"
            contested.append(dict(gene=g, hop=h, s_up=round(s_up, 3), s_down=round(s_down, 3), side=side))
        else: side = next(iter(best))
        claimed[g] = side; rec[side].append(best[side]); front[side].append(g)
COLS = ["hop", "gene", "parent", "effect", "E", "weight", "geo", "mechanism"]                  # 受容体を入力にすると、上流の木は空になる
tree = {s_: pd.DataFrame(rec[s_], columns=COLS).set_index("gene") for s_ in ("up", "down")}
assign = pd.DataFrame(contested).set_index("gene") if contested else pd.DataFrame(columns=["hop", "s_up", "s_down", "side"])
both = list(assign.index); to_up = set(assign.index[assign.side == "up"])
print(f"同じホップで両側から着いた遺伝子: {len(both)} 個 → 上流に置く {len(to_up)}、下流に置く {len(both) - len(to_up)}")
print("例（E × weight が大きい側）: " + ", ".join(f"{g}→{assign.side[g]} ({assign.s_up[g]:.2f} / {assign.s_down[g]:.2f})" for g in ['SRC', 'MTOR', 'GSK3B', 'MAPK1', 'PIK3CA', 'CDK2', 'PTK2', 'AKT1'] if g in assign.index))
print(pd.DataFrame({s_: tree[s_].groupby("hop").size() for s_ in tree}).fillna(0).astype(int).rename_axis("hop"))

## 9. 経路スコアと流れ
- 動作: 経路スコア `ps`（最良経路の `E` の積）、流れ `flow`（先の末端の `ps` の合計）、末端の数 `n_end` を計算する
- 出力: 末端の機能の内訳、流れの上位の遺伝子

In [ ]:
# 経路スコア ps、流れ flow、末端の数 n_end
for side in ("up", "down"):
    T = tree[side]
    if T.empty:                                                                              # 受容体を入力にしたときの上流（受容体より上流は解析しない）
        for c_ in ("ps", "flow", "n_end", "is_end", "func"): T[c_] = pd.Series(dtype=object)
        print(f"[{side}] 遺伝子 0（{GENE} から{'入る' if side == 'up' else '出る'}辺（geo >= {GEO_MIN}）がネットワークにない、または受容体で止めた）"); continue
    ps = {GENE: 1.0}
    for g in T.sort_values("hop").index: ps[g] = T.E[g] * ps[T.parent[g]]                    # 経路スコア = E の積
    kids = defaultdict(list)
    for g in T.index: kids[T.parent[g]].append(g)
    flow, nend = {}, {}
    for g in T.sort_values("hop", ascending=False).index:                                    # 外側（hop が大きい）から内側へ
        ch = kids.get(g, [])
        flow[g] = ps[g] if not ch else sum(flow[c] for c in ch)                              # 末端は自分の ps、そうでなければ子の流れの合計
        nend[g] = 1 if not ch else sum(nend[c] for c in ch)
    T["ps"] = pd.Series(ps); T["flow"] = pd.Series(flow); T["n_end"] = pd.Series(nend); T["is_end"] = [g not in kids for g in T.index]
    T["func"] = genes.func.reindex(T.index).fillna("other")
    ends = T[T.is_end]
    print(f"[{side}] 遺伝子 {len(T)} | 末端 {len(ends)} 個 → 機能の内訳: {ends.func.value_counts().to_dict()}")
    print(f"        流れの上位: " + ", ".join(f"{g} ({T.flow[g]:.2f}, 末端 {int(T.n_end[g])})" for g in T.flow.sort_values(ascending=False).index[:8]))

## 10. 描く遺伝子を選ぶ
- 動作: 流れの大きい順に `n_mid` 個（上流・下流それぞれ）+ その隣の遺伝子のうち経路スコアの大きい順に `n_end` 個を選ぶ。さらに上流は、終点である受容体を経路スコアの大きい順に `n_receptors` 個足す（その受容体までの経路の遺伝子も入れて入力遺伝子につなぐ。受容体は先がないので `flow` が小さく、足さないと選ばれない）
- 出力: 選んだ遺伝子の数

In [ ]:
# 描く遺伝子を選ぶ（上流は受容体も N_REC 個足す。流れの大きい順 N_MID 個 + 隣で経路スコアの大きい N_END 個）
sel = {}
for side in ("up", "down"):
    T = tree[side]
    mid = list(T.flow.sort_values(ascending=False).index[:N_MID])
    base_set = set(mid) | {GENE}
    ext = sorted([g for g in T.index if g not in base_set and T.parent[g] in base_set], key=lambda g: -T.ps[g])[:N_END]
    sel[side] = mid + ext
    if side == "up" and len(T):                                                                # 上流の終点は受容体: 経路スコア ps の大きい受容体を N_REC 個足す（その受容体までの経路の遺伝子も入れて、入力遺伝子までつなぐ）
        for g in T[T.func == "receptor"].ps.sort_values(ascending=False).index[:N_REC]:
            while g != GENE and g not in sel[side]: sel[side].append(g); g = T.parent[g]
    print(f"[{side}] 描く遺伝子 {len(sel[side])}（流れで {len(mid)} + 隣の末端側で {len(ext)}）")

## 10b. リガンドを足す（表示のためだけ）
- 動作: 上流で選んだ受容体（入力遺伝子が受容体のときは入力遺伝子）に結合する・作用するリガンド（UniProt の分泌タンパク質と、SIGNOR の低分子）を、`geo >= geo_min` の辺のうち `E` の大きい順に `max_ligands_per_receptor` 個まで足す。**選ぶ基準（`flow`・経路スコア）には入れない**（選んだあとに足す）。図の左端の列に出る
- 出力: 足したリガンドの数、受容体ごとの例

In [ ]:
# 上流の受容体に、表示のためのリガンドを足す（選択のあとなので flow・選ぶ遺伝子には影響しない）
MAX_LIG = cfg["flow"]["max_ligands_per_receptor"]
T = tree["up"]
recs = [g for g in sel["up"] if g in receptors] + ([GENE] if GENE in receptors else [])      # 上流で選んだ受容体 + 受容体である入力遺伝子
sm_names = {n for n, t in sm_type.items() if t == "smallmolecule"}                                  # 低分子（ドーパミンなど）も、受容体に作用するリガンドとして足す
cand = ok_edges[ok_edges.src.isin(ligands | sm_names) & ok_edges.dst.isin(recs) & ~ok_edges.src.isin(T.index)]       # 下流の木にあるリガンドも、表示のために上流にも置く（別の点になる）.sort_values("E", ascending=False)
added = {}
for r in cand.itertuples():
    if r.src in added or sum(1 for v in added.values() if v == r.dst) >= MAX_LIG: continue     # 1 つのリガンドは 1 回だけ。受容体ごとに最大 MAX_LIG 個
    added[r.src] = r.dst
    p_ps = 1.0 if r.dst == GENE else T.ps[r.dst]
    T.loc[r.src, ["hop", "parent", "effect", "E", "weight", "geo", "mechanism", "ps", "flow", "n_end", "is_end", "func"]] = [(1 if r.dst == GENE else int(T.hop[r.dst]) + 1), r.dst, r.effect, r.E, r.weight, r.geo, r.mechanism, p_ps * r.E, p_ps * r.E, 1, True, "small molecule" if r.src in sm_names else "ligand"]
    sel["up"].append(r.src)
T["hop"] = T.hop.astype(int); T["n_end"] = T.n_end.astype(int); T["is_end"] = T.is_end.astype(bool)
print(f"リガンドを足した受容体: {len(set(added.values()))} / 受容体 {len(recs)} | 足したリガンド {len(added)}: " + ", ".join(f"{l}→{rc}" for l, rc in list(added.items())[:10]))

## 11. 図を描く
- 動作: 左から 上流の hop | 入力遺伝子 | 下流の hop の順に並べ、辺を曲線で描く。上流のリガンドは左端、受容体はその右、下流の転写因子は右端に置く（端に置いた遺伝子の先は描かない）。点の大きさ = DepMap の相関の大きさ、縁は同じ、色 = 機能。辺の色・線種 = 符号、太さ = 証拠 `E`。**フォーカス遺伝子**（`config.yaml` の `focus.path` のリスト）は、名前を太字、丸を青にし（太い紺の縁）、名前を紺の太字にし、辺の太さと色は変えない（選ぶ基準には使わない。選ばれなかった遺伝子は図に出ない）
- 出力: 図（画面、`results/flow_view_<遺伝子名>.html` と `.png`）

In [ ]:
# 図を描く（Plotly）
FUNC_COL = {"ligand": "#79b9a8", "receptor": "#5d84b8", "transcription factor": "#e0b84f", "kinase": "#d98c5f", "phosphatase": "#9b7cb6", "ubiquitin pathway": "#a7855a", "small molecule": "#b4b4b4", "chemical": "#b4b4b4", "other": "#c4c4c4", "focus gene": "#1e5bd8"}
KINDS = {1: ("activation", "#7f95cc", "solid"), -1: ("inhibition", "#d17a77", "solid"), 0: ("unknown sign", "#8a8a8a", "dot")}
key = lambda g, side: GENE if g == GENE else f"{g}@{side[0]}"                               # 点のキー（上流は @u、下流は @d。両側に出る遺伝子は別の点になる）
base = lambda k: k.split("@")[0]

func_of = lambda g: genes.func.get(g, "other")
edge_col = {}                                                                               # 端の列に置く遺伝子（先に遺伝子が続くかどうかに関わらず、種類で決める）: 上流のリガンド・受容体は左端、下流の TF は右端
for g in sel["up"]:
    if func_of(g) == "ligand" or g in added: edge_col[(g, "up")] = "ligands"      # 表示のために足したリガンド（低分子を含む）は左端
    elif func_of(g) == "receptor": edge_col[(g, "up")] = "receptors"
for g in sel["down"]:
    if func_of(g) == "transcription factor": edge_col[(g, "down")] = "transcription factors"
for side in tree:                                                                           # 端に置いた遺伝子の先は描かない（端 = フローの終点）。例外: 受容体の先のリガンド（左端の列）
    T = tree[side]; drop = set()
    for g in sel[side]:
        p = T.parent[g]
        while p != GENE:
            if (p, side) in edge_col and not (side == "up" and edge_col.get((g, side)) == "ligands" and edge_col[(p, side)] == "receptors"): drop.add(g); break
            p = T.parent[p]
    sel[side] = [g for g in sel[side] if g not in drop]
    for g in drop: edge_col.pop((g, side), None)
hop_cols = lambda side: sorted({tree[side].hop[g] for g in sel[side] if (g, side) not in edge_col}, reverse=(side == "up"))
cols = [("ligands", [key(g, "up") for g in sel["up"] if edge_col.get((g, "up")) == "ligands"]), ("receptors", [key(g, "up") for g in sel["up"] if edge_col.get((g, "up")) == "receptors"])]
cols += [(f"up hop {h}", [key(g, "up") for g in sel["up"] if (g, "up") not in edge_col and tree["up"].hop[g] == h]) for h in hop_cols("up")]
cols += [("seed", [GENE])]
cols += [(f"down hop {h}", [key(g, "down") for g in sel["down"] if (g, "down") not in edge_col and tree["down"].hop[g] == h]) for h in hop_cols("down")]
cols += [("transcription factors", [key(g, "down") for g in sel["down"] if edge_col.get((g, "down")) == "transcription factors"])]
cols = [(n, ks) for n, ks in cols if ks]
nsubs = [max(1, -(-len(ks) // WRAP_AT)) for n, ks in cols]                                  # 各 hop の列を何本に分けるか
xbase = [0.0]                                                                                # 各 hop の中心の x（分けた列の本数だけ、隣の hop との間を広げる）
for ci in range(1, len(cols)): xbase.append(xbase[-1] + 1.0 + 0.5 * 0.6 * ((nsubs[ci - 1] - 1) + (nsubs[ci] - 1)))
x = {k: xbase[ci] for ci, (n, ks) in enumerate(cols) for k in ks}
seed_ci = [n for n, ks in cols].index("seed")

edges_d = {}                                                                                # (a, b) -> (E, 符号)。a, b は点のキー。図では左から右へ向かう
for side in ("up", "down"):
    T, S = tree[side], set(sel[side])
    for g in S:                                                                             # 最良経路の辺
        p = T.parent[g]
        a, b = (key(g, side), key(p, side)) if side == "up" else (key(p, side), key(g, side))
        if x[a] < x[b]: edges_d[(a, b)] = (T.E[g], T.effect[g])
    ex = ok_edges[ok_edges.src.isin(S | {GENE}) & ok_edges.dst.isin(S | {GENE}) & (ok_edges.src != ok_edges.dst)]
    ex = ex[np.array([x.get(key(s, side), 9e9) < x.get(key(d, side), -9e9) for s, d in zip(ex.src, ex.dst)], dtype=bool)]            # 左から右へ進む辺だけ
    for r in ex.sort_values("E", ascending=False).groupby("src").head(MAX_OUT).itertuples():
        edges_d.setdefault((key(r.src, side), key(r.dst, side)), (r.E, r.effect))

y = {GENE: 0.0}
for ci in range(seed_ci - 1, -1, -1):                                                       # 上流側: 入力遺伝子に近い列から外へ
    ks = cols[ci][1]
    for i, k in enumerate(sorted(ks, key=lambda k: (np.mean([y[b] for (a, b) in edges_d if a == k and b in y] or [1e9]), k))): y[k] = (len(ks) - 1) / 2 - i
for ci in range(seed_ci + 1, len(cols)):                                                    # 下流側
    ks = cols[ci][1]
    for i, k in enumerate(sorted(ks, key=lambda k: (np.mean([y[a] for (a, b) in edges_d if b == k and a in y] or [1e9]), k))): y[k] = (len(ks) - 1) / 2 - i
for ci, (n, ks) in enumerate(cols):                                                         # 1 つの列の遺伝子が WRAP_AT を超えるときは、ceil(遺伝子数 / WRAP_AT) 本の列に分ける
    nsub = -(-len(ks) // WRAP_AT)
    if nsub < 2: continue
    order = sorted(ks, key=lambda k: -y[k]); gap = 0.6
    for j, part in enumerate(np.array_split(order, nsub)):                             # 各列の遺伝子の数は、差が 1 以内になるように分ける
        part = list(part)
        for i, k in enumerate(part): x[k] = xbase[ci] + (j - (nsub - 1) / 2) * gap; y[k] = (len(part) - 1) / 2 - i

rows_needed = lambda n: -(-n // (-(-n // WRAP_AT)))                                       # 分けたあとの 1 列の行数
ncol, nmax = len(cols), max(rows_needed(len(ks)) for n, ks in cols)
xr = (min(x.values()) - 0.5, max(x.values()) + 0.8)
W, Hpx = max(1500, int(175 * (xr[1] - xr[0]))), max(700, ROW_PX * nmax + 240)
yr = (-nmax / 2 - 1.5, nmax / 2 + 2.5)
sx, sy = (W - 40) / (xr[1] - xr[0]), (Hpx - 140) / (yr[1] - yr[0])

fig = go.Figure()
first_leg = set()
for e_, (kname, kcol, kdash) in KINDS.items():
    for wcond, w_, foc in ((lambda e: e >= 0.8, 2.0, False), (lambda e: 0.6 <= e < 0.8, 1.6, False), (lambda e: 0.4 <= e < 0.6, 1.2, False), (lambda e: 0.2 <= e < 0.4, 0.9, False), (lambda e: e < 0.2, 0.6, False)):                    # 証拠 E で 5 段階の太さ
        xs, ys, hx, hy, hang = [], [], [], [], []
        kc = kcol
        for (a, b), (E_, ef) in edges_d.items():
            if ef != e_ or not wcond(E_): continue
            (x0, y0), (x1, y1) = (x[a], y[a]), (x[b], y[b]); dx = x1 - x0
            P = np.array([[x0, y0], [x0 + 0.5 * dx, y0], [x1 - 0.5 * dx, y1], [x1, y1]])
            pt = lambda t: (1 - t) ** 3 * P[0] + 3 * (1 - t) ** 2 * t * P[1] + 3 * (1 - t) * t ** 2 * P[2] + t ** 3 * P[3]
            dv = lambda t: 3 * (1 - t) ** 2 * (P[1] - P[0]) + 6 * (1 - t) * t * (P[2] - P[1]) + 3 * t ** 2 * (P[3] - P[2])
            pts = np.array([pt(t) for t in np.linspace(0, 1, 28)]); xs += list(pts[:, 0]) + [None]; ys += list(pts[:, 1]) + [None]
            te = 0.93 if e_ == -1 else 0.88; p_, v_ = pt(te), dv(te); th = math.degrees(math.atan2(v_[0] * sx, v_[1] * sy))
            hx.append(p_[0]); hy.append(p_[1]); hang.append(th - 90 if e_ == -1 else th)
        if not xs: continue
        showleg = kname not in first_leg; first_leg.add(kname)
        fig.add_trace(go.Scatter(x=xs, y=ys, mode="lines", line=dict(color=kc, width=w_, dash=kdash), opacity=0.75, hoverinfo="skip", name=f"{kname} edge", legendgroup=kname, showlegend=showleg))
        fig.add_trace(go.Scatter(x=hx, y=hy, mode="markers", hoverinfo="skip", showlegend=False, legendgroup=kname,
                                 marker=dict(symbol="line-ns" if e_ == -1 else "arrow", size=7 + 2 * w_, angle=hang, angleref="up", color=kc, line=dict(color=kc, width=2.5) if e_ == -1 else None)))
term = {}                                                                                    # 末端の遺伝子（先に描く遺伝子がない）: 点のキー -> 流れのスコア（末端は自分の経路スコア、端の列の遺伝子はその先の末端の合計）
for side in tree:
    ch = {tree[side].parent[g] for g in sel[side]}
    for g in sel[side]:
        if g not in ch: term[key(g, side)] = (tree[side].flow[g], side)
allk = [k for k in x if k != GENE]
print(f"フォーカス遺伝子 {len(FOCUS)} 個 | 図に描いた: " + ", ".join(sorted(g for g in FOCUS if any(base(k) == g for k in allk))) + " | 図にない: " + ", ".join(sorted(g for g in FOCUS if not any(base(k) == g for k in allk))))
for cat in [c_ for c_ in FUNC_COL if c_ != "focus gene"] + ["focus gene"]:
    ks = [k for k in allk if (base(k) in FOCUS) == (cat == "focus gene") and (cat == "focus gene" or genes.func.get(base(k), "other") == cat)]
    if not ks: continue
    r_ = [corr.get(base(k), np.nan) for k in ks]
    sz = [8 + 70 * (abs(v) if v == v else 0) for v in r_]
    if cat == 'focus gene': sz = [s_ + 6 for s_ in sz]                                    # フォーカス: 少し大きく
    if cat in ('small molecule', 'chemical'): sz = [6] * len(ks)                               # 低分子: 小さく固定、灰色（相関の大きさは使わない）
    fig.add_trace(go.Scatter(x=[x[k] for k in ks], y=[y[k] for k in ks], mode="markers", marker=dict(size=sz, color=FUNC_COL[cat], symbol="square" if cat in ("small molecule", "chemical") else "circle", line=dict(color="#0a2463", width=3.5) if cat == "focus gene" else dict(color="#444444", width=1.5)), name=cat,
                             hovertext=[f"<b>{base(k)}</b><br>{genes.gene_name.get(base(k), '')}<br>{TISSUE} nTPM: {genes.ntpm.get(base(k), float('nan')):.1f}<br>{genes.func.get(base(k), 'other') if cat == 'focus gene' else cat}<br>DepMap r with {GENE}: {('%.2f' % v) if v == v else 'n/a'}" for k, v in zip(ks, r_)], hoverinfo="text"))
    for k, s in zip(ks, sz):                                                                # 名前は自分の点の上部（点の上端に少し重なる位置）
        if k in term: fig.add_annotation(x=x[k], y=y[k], text=f'{term[k][0]:.2g}', showarrow=False, xanchor='right' if term[k][1] == 'up' else 'left', yanchor='middle', xshift=-(s / 2 + 3) if term[k][1] == 'up' else s / 2 + 3, font=dict(size=10, color='#777777'))   # 末端に流れのスコア（上流は点の左、下流は点の右。辺が来ない側）
        fig.add_annotation(x=x[k], y=y[k], text=("<b>" + base(k) + "</b>" if cat == "focus gene" else (base(k) if len(base(k)) <= 18 else base(k)[:17] + "…")), showarrow=False, bgcolor="rgba(255,255,255,0.4)", borderpad=0, xanchor="center", yanchor="bottom", yshift=(max(s / 2 - 3, 2) if s <= 20 else s / 2 + 1), font=dict(size=10, color="#222222") if cat in ("small molecule", "chemical") else dict(size=14, color="#0a2463" if cat == "focus gene" else "#222222"))
fig.add_annotation(x=xr[0] + 0.1, y=yr[0] + 0.6, text=f"tissue: {TISSUE} (genes with nTPM >= {MIN_NTPM}; edge score x expression factor)  |  circle size: |DepMap r| with {GENE}  |  edge width: evidence E (thin 0.6 to thick 2)  |  number beside end genes = flow score", showarrow=False, xanchor="left", font=dict(size=11, color="#666666"))   # 凡例は最小限。大きさと太さは概要だけを書く
fig.add_trace(go.Scatter(x=[x[GENE]], y=[0], mode="markers+text", text=[GENE], textposition="top center", textfont=dict(size=15), name="seed",
                         marker=dict(size=22, color="#555555", symbol="diamond"), hoverinfo="skip"))
for ci, (n, ks) in enumerate(cols): fig.add_annotation(x=xbase[ci], y=yr[1] - 0.6, text=n, showarrow=False, font=dict(size=13, color="#666666"))
fig.update_layout(width=W, height=Hpx, plot_bgcolor="white", margin=dict(l=20, r=20, t=30, b=170), xaxis=dict(visible=False, range=list(xr)), yaxis=dict(visible=False, range=list(yr)),
                  legend=dict(orientation="h", yanchor="top", y=-0.01, xanchor="center", x=0.5, font=dict(size=11)))
fig.write_html(f"{OUT}/flow_view_{GENE}.html", include_plotlyjs="cdn")
try: fig.write_image(f"{OUT}/flow_view_{GENE}.png", scale=1.4)
except Exception as e: print("PNG は保存できなかった:", str(e)[:80])
print(f"描いた点 {len(x) - 1} 個 | 辺 {len(edges_d)} 本")
fig.show()

## 12. 遺伝子の表を保存する
- 動作: 上流・下流の全遺伝子について、経路・符号・経路スコア・流れ・末端の数・機能・DepMap の相関・図に描いたかを表にして保存する
- 出力: 流れの大きい順の上位 30 遺伝子の表、`results/flow_view_<遺伝子名>_genes.csv`

In [ ]:
# 遺伝子の表
rows = []
for side in ("up", "down"):
    T = tree[side]
    for g in T.index:
        p, sg, q = [g], T.effect[g], T.parent[g]
        while q != GENE: p.append(q); sg *= T.effect[q]; q = T.parent[q]
        path = " > ".join(p + [GENE]) if side == "up" else " > ".join([GENE] + p[::-1])
        rows.append(dict(gene=g, gene_name=genes.gene_name.get(g, ""), tissue_nTPM=genes.ntpm.get(g, np.nan), tissue_tau=genes.tau.get(g, np.nan), side=side, hop=int(T.hop[g]), parent=T.parent[g], path=path, path_sign={1: "activation", -1: "inhibition", 0: "unknown"}[sg],
                         path_score=round(T.ps[g], 4), flow=round(T.flow[g], 4), n_end=int(T.n_end[g]), is_end=bool(T.is_end[g]), E_to_parent=round(T.E[g], 3), geo_to_parent=round(T.geo[g], 3),
                         mechanism=T.mechanism[g], function=T.func[g], locations=genes.locations.get(g, ""), uniprot_flags=genes.uniprot_flags.get(g, ""),
                         depmap_r=round(corr.get(g, np.nan), 3), drawn=g in set(sel[side]), in_both_sides_before=g in assign.index, s_up=assign.s_up.get(g, np.nan), s_down=assign.s_down.get(g, np.nan)))
tab = pd.DataFrame(rows).sort_values(["flow", "path_score"], ascending=False).reset_index(drop=True)
tab.to_csv(f"{OUT}/flow_view_{GENE}_genes.csv", index=False, encoding="utf-8-sig")
print(f"保存: {OUT}/flow_view_{GENE}_genes.csv | {len(tab)} 行（遺伝子 × 側）")
pd.set_option("display.width", 250); pd.set_option("display.max_columns", 30); pd.set_option("display.max_colwidth", 60)
tab[["gene", "gene_name", "side", "hop", "parent", "path_sign", "path_score", "flow", "n_end", "is_end", "function", "depmap_r", "drawn"]].head(30)